# PLM ELF, CDO/RCDO and PDI audit

These cells use **recorded hardware evidence** by default. They do not program a board.
Use the configured Python (SCU35 PMC) kernel. Fresh hardware commands are shown in Markdown and must be invoked explicitly after configuring your exact SCU35 and recovery image.
Current PL timing uses AXI Timer; native PMC timing uses `mcycle`. JTAG/UART are outside operation timing.

The PDI contains loadable PLM bytes, configuration data and headers/hash framing. An ELF container can be much larger than its loadable segments. The retained normal PDI breakdown is from the older boot-design revision, not the new timer build.

Using the installed binaries from your configuration, run `bootgen -arch spartanup -read /path/to/demo.pdi` and `riscv64-unknown-elf-readelf -h -l /path/to/plm.elf`. These are offline inspections. Neither a readable PDI nor DONE status alone proves authentication.

In [1]:
from pathlib import Path
import hashlib, json, sys, time
import pandas as pd
# Works from the notebook directory, lab root, or parent repository.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for base in candidates for p in (base, base / "scu35") if (p / "lab.py").is_file()), None)
assert ROOT is not None, "Launch Jupyter inside this checkout or its scu35 directory"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "host"))
from compare import parse_capture
from lab import config
capture = parse_capture(ROOT / "evidence/timer/uart.txt", "7d1bf77bc454dd1f")
rows = pd.DataFrame(capture["rows"])
print("Recorded board capture: 31 complete rows; current Python:", sys.version.split()[0])


Recorded board capture: 31 complete rows; current Python: 3.11.9


In [2]:
breakdown = json.loads((ROOT / "evidence/pdi/breakdown.json").read_text())
display(breakdown)
bd = json.loads((ROOT / "hw/sys.bd").read_text())
segments = bd["design"]["addressing"]["/mb"]["address_spaces"]["Data"]["segments"]
display(pd.DataFrame(segments).T[["address_block", "offset", "range"]])
assert any(s["offset"].lower() == "0x40000000" for s in segments.values())

{'source': 'C:\\aiup\\sec35\\compare\\compare.pdi',
 'pdi_sha256': '12fa0536cf8e8301b324972031c23959c6da82ca1301867d8b7b99cd12f64fce',
 'total_bytes': 1359840,
 'part': 'xcsu35p-sbvb625-2-e',
 'vendor_utility': 'AMD Bootgen 2026.1, -arch spartanup -read / -dump boot_files',
 'file_layout': [{'name': 'Boot headers/root hashes/padding',
   'offset': 0,
   'bytes': 976},
  {'name': 'PLM encoded partition', 'offset': 976, 'bytes': 26144},
  {'name': 'FPGA CDO encoded partition', 'offset': 27120, 'bytes': 1332720}],
 'breakdown': [{'name': 'PLM firmware payload',
   'bytes': 26048,
   'KiB': 25.4375,
   'percent': 1.915519472879162},
  {'name': 'FPGA configuration stream',
   'bytes': 1326152,
   'KiB': 1295.0703125,
   'percent': 97.52264972349688},
  {'name': 'CDO commands / framing',
   'bytes': 1384,
   'KiB': 1.3515625,
   'percent': 0.1017766796093658},
  {'name': 'Partition SHA3 chain bytes',
   'bytes': 5280,
   'KiB': 5.15625,
   'percent': 0.3882809742322626},
  {'name': 'Boot hea

,address_block,offset,range
SEG_dlmb_bram_if_cntlr_Mem,/mb_local_memory/dlmb_bram_if_cntlr/SLMB/Mem,0x00000000,128K
SEG_pmc_ccu_apb_0,/pmc/CCU_APB/ccu_apb_0,0x04130000,64K
SEG_pmc_ccu_rd_stream_0,/pmc/CCU_RD_STREAM/ccu_rd_stream_0,0x04060000,64K
SEG_pmc_ccu_wr_stream_0,/pmc/CCU_WR_STREAM/ccu_wr_stream_0,0x04070000,64K
SEG_pmc_plm_config_0,/pmc/PLM_CONFIG/plm_config_0,0x04020000,64K
SEG_pmc_pmcl_aes_0,/pmc/PMC_AES/pmcl_aes_0,0x040F0000,64K
SEG_pmc_pmcl_dma_0,/pmc/PMCL_DMA/pmcl_dma_0,0x040E0000,64K
SEG_pmc_pmcl_efuse_0,/pmc/PMCL_EFUSE/pmcl_efuse_0,0x04160000,64K
SEG_pmc_pmcl_global_0,/pmc/PMCL_GLOBAL/pmcl_global_0,0x040A0000,128K
SEG_pmc_pmcl_ospi_0,/pmc/PMCL_OSPI/pmcl_ospi_0,0x04090000,64K


## Fresh hardware: explicit step

Configure `config.json` first. Run `python lab.py hardware --pdi /path/to/build/demo.pdi` from the lab directory for the full runtime suite, or the native/boot command above for that path. Save logs, verify recovery and compare results before discussing speed. See [audit checkpoints](../docs/audit.md).